In [14]:

import pandas as pd
import glob

files = sorted(glob.glob("data/**/financials-annual*.csv.gz", recursive=True))
print("Found:", files)

bq = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print(bq.shape)
print(bq.columns.tolist())

Found: ['data/financials-annual_0_0_0.csv.gz', 'data/financials-annual_1_0_0.csv.gz']
(68109, 65)
['BQ_ACCOUNTS_PAYABLE', 'BQ_ASSET_TURNOVER', 'BQ_BUILDINGS_AND_OTHER_DEPRECIABLE_ASSETS', 'BQ_CASH', 'BQ_COMPANY_NAME', 'BQ_COR', 'BQ_COR_PER_EMP', 'BQ_CURRENT_ASSETS', 'BQ_CURRENT_LIABILITIES', 'BQ_CURRENT_RATIO', 'BQ_DEBT_TO_EQUITY_RATIO', 'BQ_DEPLETABLE_ASSETS', 'BQ_EBITDA', 'BQ_EBITDA_MARGIN', 'BQ_EBITDA_PER_EMP', 'BQ_FISCAL_YEAR_END', 'BQ_GROSS_PROFIT', 'BQ_GROSS_PROFIT_MARGIN', 'BQ_GROSS_PROFIT_PER_EMP', 'BQ_ID', 'BQ_INTANGIBLE_ASSETS_AMORTIZABLE', 'BQ_INVENTORIES', 'BQ_LAND', 'BQ_LESS_ACCUMULATED_AMORTIZATION', 'BQ_LESS_ACCUMULATED_DEPLETION', 'BQ_LESS_ACCUMULATED_DEPRECIATION', 'BQ_LESS_ALLOWANCE_FOR_BAD_DEBTS', 'BQ_LOANS_FROM_SHAREHOLDERS', 'BQ_LOANS_TO_SHAREHOLDERS', 'BQ_LONG_TERM_DEBT', 'BQ_MORTGAGE_AND_REAL_ESTATE_LOANS', 'BQ_NET_INCOME', 'BQ_NET_INCOME_PER_EMP', 'BQ_NET_PROFIT_MARGIN', 'BQ_NON_CURRENT_ASSETS', 'BQ_NON_CURRENT_LIABILITIES', 'BQ_OPERATING_EXPENSES', 'BQ_OPERATIN

In [15]:
import numpy as np

d = bq.copy()
d.columns = [c.lower().replace("bq_", "") for c in d.columns]
cols = ["ebitda", "operating_income", "buildings_and_other_depreciable_assets", "depletable_assets",
        "land", "less_accumulated_depreciation", "less_accumulated_depletion",
        "current_assets", "cash", "current_liabilities", "short_term_debt"]
for c in cols:
    d[c] = pd.to_numeric(d[c], errors="coerce")
d = d.sort_values(["id", "year"]).drop_duplicates(["id", "year"])

# Derived items, from balance sheet columns only
d["gross_ppe"] = (d.buildings_and_other_depreciable_assets
                  + d.depletable_assets.fillna(0) + d.land.fillna(0))
d["acc_dep"] = d.less_accumulated_depreciation.abs()
d["nwc"] = (d.current_assets - d.cash.fillna(0)) - (d.current_liabilities - d.short_term_debt.fillna(0))

prev = d.groupby("id")[["year", "gross_ppe", "acc_dep", "nwc"]].shift(1)
ok = prev.year == d.year - 1
d["capex"] = np.where(ok, d.gross_ppe - prev.gross_ppe, np.nan)   # missing in either year -> NaN
d["depr"]  = np.where(ok, d.acc_dep - prev.acc_dep, np.nan)
d["d_nwc"] = np.where(ok, d.nwc - prev.nwc, np.nan)

pos = d[d.ebitda > 0].copy()
pos["da_pct"] = pos.depr / pos.ebitda                         # depreciation only
pos["capex_nwc_pct"] = (pos.capex + pos.d_nwc) / pos.ebitda

# One typical value per company (median across its years), to avoid year-to-year noise
per_co = pos.groupby("id")[["da_pct", "capex_nwc_pct"]].median()
per_co = per_co[per_co.da_pct.between(0, 0.8) & per_co.capex_nwc_pct.between(-0.5, 1.5)]

# EBITDA growth: 5-year CAGR, positive EBITDA at both ends
HOLD = 5
e = d[["id", "year", "ebitda"]].dropna()
fwd = e.assign(year=e.year - HOLD)
pairs = e.merge(fwd, on=["id", "year"], suffixes=("", "_fwd"))
pairs = pairs[(pairs.ebitda > 0) & (pairs.ebitda_fwd > 0)]
growth = ((pairs.ebitda_fwd / pairs.ebitda) ** (1 / HOLD) - 1)
growth = growth[growth.between(-0.30, 0.50)]

summary = {"ebitda_growth": growth, "da_pct": per_co.da_pct, "capex_nwc_pct": per_co.capex_nwc_pct}
stats = pd.DataFrame({k: {"n": len(s), "p10": s.quantile(.1), "p50": s.quantile(.5), "p90": s.quantile(.9)}
                      for k, s in summary.items()}).T
print(stats.round(3))

import os
os.makedirs("results", exist_ok=True)
stats.to_csv("results/brightquery_ranges.csv")

                     n    p10    p50    p90
ebitda_growth  39848.0 -0.063  0.053  0.242
da_pct          2539.0  0.016  0.091  0.337
capex_nwc_pct   2539.0 -0.161  0.105  0.677


In [16]:
d["revenue"] = pd.to_numeric(d["revenue"], errors="coerce")
d["margin"] = d.ebitda / d.revenue

screen = d.groupby("id").agg(
    n_years=("year", "count"),
    always_pos=("ebitda", lambda s: (s > 0).all()),
    margin=("margin", "median"),
)
keep = screen[(screen.n_years >= 5) & screen.always_pos & (screen.margin >= 0.10)].index
co_pe = co[co.index.isin(keep)]
co_pe.to_csv("data/brightquery_company_params_pe.csv")

print(f"All companies: {len(co)}   PE-style screened: {len(co_pe)}")
print("\nAll companies:\n", co.describe().round(3))
print("\nScreened:\n", co_pe.describe().round(3))

All companies: 773   PE-style screened: 562

All companies:
             da  capex_nwc  ebitda_g
count  773.000    773.000   773.000
mean     0.387      0.456     0.055
std      0.216      0.385     0.087
min      0.001     -0.494    -0.230
25%      0.201      0.196     0.006
50%      0.374      0.444     0.043
75%      0.565      0.694     0.088
max      0.795      1.500     0.411

Screened:
             da  capex_nwc  ebitda_g
count  562.000    562.000   562.000
mean     0.383      0.445     0.050
std      0.211      0.369     0.084
min      0.001     -0.484    -0.230
25%      0.200      0.195     0.007
50%      0.368      0.423     0.042
75%      0.547      0.676     0.080
max      0.795      1.500     0.408


In [17]:
pairs["g"] = (pairs.ebitda_fwd / pairs.ebitda) ** (1 / HOLD) - 1
g = pairs[pairs.g.between(-0.30, 0.50)].groupby("id").g.median().rename("ebitda_g")

co = (per_co.rename(columns={"da_pct": "da", "capex_nwc_pct": "capex_nwc"})
            .join(g, how="inner").dropna())
co.to_csv("data/brightquery_company_params.csv")

print(len(co), "companies")
print(co.describe().round(3))
print(co.corr(method="spearman").round(2))   # real correlations between the inputs

2470 companies
             da  capex_nwc  ebitda_g
count  2470.000   2470.000  2470.000
mean      0.143      0.186     0.079
std       0.146      0.350     0.104
min       0.000     -0.500    -0.269
25%       0.043     -0.017     0.016
50%       0.092      0.105     0.058
75%       0.195      0.309     0.125
max       0.787      1.482     0.499
             da  capex_nwc  ebitda_g
da         1.00       0.26      0.09
capex_nwc  0.26       1.00      0.19
ebitda_g   0.09       0.19      1.00


In [18]:
# How many companies did each filtering step remove?
per_co_all = pos.groupby("id")[["da_pct", "capex_nwc_pct"]].median()   # before outlier filter
n_total = len(per_co_all)
n_missing = per_co_all.isna().any(axis=1).sum()                          # not enough data to calculate
n_outliers = (n_total - n_missing) - len(per_co)                         # removed by our bounds

print(f"Companies with positive EBITDA:     {n_total}")
print(f"  dropped, insufficient data:        {n_missing}")
print(f"  dropped, outside outlier bounds:   {n_outliers}  ({n_outliers / (n_total - n_missing):.1%})")
print(f"  kept:                              {len(per_co)}")

# Same check for EBITDA growth (counted in 5-year windows, not companies)
all_growth = (pairs.ebitda_fwd / pairs.ebitda) ** (1 / HOLD) - 1
print(f"\nGrowth windows: {len(all_growth)}, dropped as outliers: {len(all_growth) - len(growth)} "
      f"({1 - len(growth) / len(all_growth):.1%})")

Companies with positive EBITDA:     5031
  dropped, insufficient data:        1028
  dropped, outside outlier bounds:   1464  (36.6%)
  kept:                              2539

Growth windows: 41793, dropped as outliers: 1945 (4.7%)


In [19]:
pos["capex_pct"] = pos.capex / pos.ebitda
pos["dnwc_pct"] = pos.d_nwc / pos.ebitda
parts = pos.groupby("id")[["da_pct", "capex_pct", "dnwc_pct"]].median()
print(parts.describe(percentiles=[.05, .25, .5, .75, .95]).round(2))

complete = per_co_all.dropna()
print("\nD&A outside 0–80%:", (~complete.da_pct.between(0, 0.8)).sum())
print("Capex+NWC outside −50%–150%:", (~complete.capex_nwc_pct.between(-0.5, 1.5)).sum())

print("\nShare of rows with each field filled in:")
print(d[["ebitda", "operating_income", "buildings_and_other_depreciable_assets",
         "less_accumulated_depreciation", "current_assets", "current_liabilities", "cash"]]
      .notna().mean().round(2))

          da_pct  capex_pct  dnwc_pct
count    4042.00    4532.00   4768.00
mean       36.83      75.37    -28.55
std      1827.72    1693.67   1836.93
min    -34074.05   -6357.22 -87426.33
5%         -0.00      -0.49    -12.22
25%         0.04      -0.02     -0.30
50%         0.10       0.02     -0.00
75%         0.26       0.16      0.15
95%         3.27       3.18      5.49
max    109674.25   73013.92  11944.47

D&A outside 0–80%: 646
Capex+NWC outside −50%–150%: 1238

Share of rows with each field filled in:
ebitda                                    1.00
operating_income                          0.97
buildings_and_other_depreciable_assets    0.76
less_accumulated_depreciation             0.63
current_assets                            0.79
current_liabilities                       0.77
cash                                      0.78
dtype: float64
